# KBAI ARC Milestone 2: Symbolic Baseline

This notebook is intended as a pilot experiment with an independently structured symbolic agent, not a complete ARC solver.

Implemented: Generate-and-test over a small domain-specific language (DSL), plus an explicit version space of hypotheses surviving each demonstration. The DSL has pixel-level and object-level rules.

Not implemented yet: automatic incremental concept learning (ICL). Selecting among hard-coded object predicates (current implementation) is not the same as learning new concepts.

Data: This project initially uses the public ARC-AGI-1 training dataset. Each task contains input-output demonstrations illustrating an unknown transformation, along with one or more test inputs. The agent uses the demonstrations to generate and evaluate candidate transformation hypotheses while the corresponding test solutions are reserved for evaluating its predictions. Current experiments will use a subset of 25 training tasks to examine hypothesis consistency, version-space reduction, and the effects of different representations. The public evaluation dataset will be kept separate for subsequent validation. ARC-AGI-2 may be considered later to evaluate the approach on more challenging problems using ICL.

# 1. Imports and Data Loading

In [19]:
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

# Use ARC-AGI-1 dataset for development
#DATA_DIR = Path('../input/ARC-AGI-1')

# Use for submission
DATA_DIR = Path('../input/problems')

def load_json(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with path.open() as file:
        return json.load(file)

# 2. Representations and Primitive Transformations

A hypothesis is an executable rule. Pixel-level rules apply to the complete grid. Object-level rules first identify nonzero, same-color connected componenets under either 4-neighbor or 8-neighbor connectivity. Then it selects the largest or smallest component with ties broken by the top-left position.

The restricted vocabulary is a deliberate design choice to ensure feasibility of the pilot. It compares a fixed pixel-only vocabulary to pixel + object vocabulary.

In [20]:
GEOMETRIC_OPS = (
    'identity', 'rotate_90', 'rotate_180', 'rotate_270', 'flip_horizontal', 'flip_vertical', 'transpose', 'anti_transpose'
)

def as_grid(values):
    arr = np.asarray(values, dtype=np.int64)

    if arr.ndim != 2:
        raise ValueError('ARC grids must be 2D arrays')
    
    return arr

def transform_pixels(a, op):

    if op == 'identity':        return a
    if op == 'rotate_90':        return np.rot90(a, 1)
    if op == 'rotate_180':        return np.rot90(a, 2)
    if op == 'rotate_270':        return np.rot90(a, 3)
    if op == 'flip_horizontal':        return np.fliplr(a)
    if op == 'flip_vertical':        return np.flipud(a)
    if op == 'transpose':        return a.T.copy()
    if op == 'anti_transpose':        return np.rot90(a.T, 2)
    
    raise ValueError(f"Unknown geometric operation: {op}")

def connected_components(grid, connectivity=4):
    """ List of monochromatic, nonbackground objects, each a list of (row,col)"""

    height, width = grid.shape
    seen = np.zeros((height, width), dtype=bool)
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if connectivity == 8:
        neighbors += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    result = []

    for r in range(height):
        for c in range(width):
            if grid[r, c] == 0 or seen[r, c]:
                continue

            color = grid[r,c]
            seen[r,c] = True
            stack = [(r, c)]
            cells = []
            while stack:
                cr, cc = stack.pop()
                cells.append((cr, cc))
                for dr, dc in neighbors:
                    nr, nc = cr + dr, cc + dc
                    if (0 <= nr < height and 0 <= nc < width and not seen[nr, nc] and grid[nr, nc] == color):
                        seen[nr, nc] = True
                        stack.append((nr, nc))

            result.append(cells)
    return result

def select_component(components, selector):
    if not components:
        return None

    # Ties are deterministic, sorting by position is independent of BFS traversal order
    order = sorted(components, key =lambda comp: (-len(comp) if selector == 'largest' else len(comp), min(comp)))
    return order[0]

def transform_object(grid, action, selector, connnectivity, target=None):
    obj = select_component(connected_components(grid, connectivity=connnectivity), selector)
    if obj is None:
        return None

    rows, cols = zip(*obj)

    if action == 'crop':
        return grid[min(rows):max(rows)+1, min(cols):max(cols)+1].copy()
    
    if action == 'keep':
        output = np.zeros_like(grid)
        output[rows, cols] = grid[rows, cols]
    else:
        output = grid.copy()
        if action == 'erase':
            output[rows, cols] = 0
        elif action == 'recolor':
            output[rows, cols] = int(target)
        else:
            raise ValueError(f"Unknown object action: {action}")

    return output


@dataclass(frozen=True)
class Hypothesis:
    representation: str
    family: str
    arguments: tuple
    complexity: int

    @property
    def label(self):
        return f'{self.representation}:{self.family}{self.arguments}'

    def apply(self, input_grid):
        grid = as_grid(input_grid)
        if self.family == 'geometry':
            return transform_pixels(grid, self.arguments[0])
        elif self.family == 'recolor_pixel':
            source, target = self.arguments
            output = grid.copy()
            output[grid == source] = target
            return output
        elif self.family == 'object':
            action, selector, connectivity, target = self.arguments
            return transform_object(grid, action, selector, connectivity, target)
        else:
            raise ValueError(f"Unknown hypothesis family: {self.family}")

# 3. Hypothesis Generation and Version-Space Filtering

Candidates are created once per ARC task using only its demonstration pairs. Their descriptions are fixed while the version space is filtered example by example. Thus reported survival counts are monotonically non-increasing.

The object-level library is intentionally small .Its fixed selectors (largest,smallest) and connectivity conventions (4, 8) are engineered primitives that in a later implementation will be discovered by ICL.

In [21]:
def generate_hypotheses(task, include_objects=True):
    """Build a finite DSL for a task with no reference to test outputs"""
    examples = task['train']
    input_colors = set()
    output_colors = set()

    for example in examples:
        input_colors.update(np.unique(as_grid(example['input'])).tolist())
        output_colors.update(np.unique(as_grid(example['output'])).tolist())

    candidates = []
    for op in GEOMETRIC_OPS:
        complexity = 0 if op == 'identity' else 1
        candidates.append(Hypothesis('pixel', 'geometry', (op,), complexity))

    for source in sorted(input_colors):
        for target in sorted(output_colors):
            if source != target:
                candidates.append(Hypothesis('pixel', 'recolor_pixel', (source, target), 2))


    if include_objects:
        for connectivity in (4, 8):
            for selector in ('largest', 'smallest'):
                for action in ('keep', 'erase', 'crop'):
                    candidates.append(Hypothesis(
                        f'objects_{connectivity}', 'object',
                        (action, selector, connectivity, None), 3
                    ))
                for target in sorted(output_colors):
                    candidates.append(Hypothesis(
                        f'objects_{connectivity}', 'object',
                        ('recolor', selector, connectivity, target), 4
                    ))

    return candidates

def matches_example(hypothesis, example):
    predicted = hypothesis.apply(example['input'])
    return predicted is not None and np.array_equal(predicted, as_grid(example['output']))

def compute_version_space(task, include_objects=True):
    """Return surviving hypotheses and counts after each demonstration"""
    candidates = generate_hypotheses(task, include_objects)
    history = []
    for index, example in enumerate(task['train'], start=1):
        candidates = [h for h in candidates if matches_example(h, example)]
        history.append({
            'examples_seen': index,
            'total_survivors': len(candidates),
            'pixel_survivors': sum(h.representation == 'pixel' for h in candidates),
            'object_survivors': sum(h.representation.startswith('objects_') for h in candidates),
        })
    candidates.sort(key=lambda h: (h.complexity, h.representation, h.label))
    return candidates, history

def valid_arc_prediction(grid):
        return (grid is not None and grid.ndim == 2 and
            1 <= grid.shape[0] <= 30 and 1 <= grid.shape[1] <= 30 and
            np.all((grid >= 0) & (grid <= 9)))


def predict_two(test_input, version_space):
    """Prefer simpler and distinct predictions. Always return two attempts."""
    predictions = []
    for hypothesis in version_space:
        guess = hypothesis.apply(test_input)
        if valid_arc_prediction(guess) and not any(np.array_equal(guess, p) for p in predictions):
            predictions.append(guess)
            if len(predictions) == 2:
                break
    if not predictions:
        predictions = [as_grid(test_input).copy()]  # valid, explicit fallback; not a learned rule
    if len(predictions) == 1:
        predictions.append(predictions[0].copy())
    return {'attempt_1': predictions[0].tolist(), 'attempt_2': predictions[1].tolist()}


def solve_task(task, include_objects=True):
    consistent, history = compute_version_space(task, include_objects)
    answers = [predict_two(item['input'], consistent) for item in task['test']]
    return answers, consistent, history

# 4. Small Synthetic Tests (without dataset files)

This is to tests incremental elimination of inconsistent rules before benchmarking ARC performance.

In [22]:
ambiguous_task = {
    'train': [
        {'input': [[1,0],[0,1]], 'output': [[1,0],[0,1]]},
        {'input': [[1,0],[0,0]], 'output': [[0,0],[0,1]]},
    ],
    'test': [{'input': [[2,0],[0,0]]}],
}
answers, survivors, history = solve_task(ambiguous_task)
assert history[0]['total_survivors'] > history[1]['total_survivors']
assert any(h.label == "pixel:geometry('rotate_180',)" for h in survivors)
assert answers[0]['attempt_1'] == [[0,0],[0,2]] or answers[0]['attempt_2'] == [[0,0],[0,2]]
assert set(answers[0]) == {'attempt_1', 'attempt_2'}
assert all(len(a) > 0 for a in answers[0].values())

# Verify the pixel transformation library and object extraction.
a = np.array([[1,2,3],[4,5,6]])
assert np.array_equal(transform_pixels(a, 'rotate_180'), np.array([[6,5,4],[3,2,1]]))
assert len(connected_components(np.array([[1,0,1],[0,1,0]]), connectivity=4)) == 3
assert len(connected_components(np.array([[1,0,1],[0,1,0]]), connectivity=8)) == 1
print('Synthetic checks passed. Version-space history:', history)
print('Surviving example hypotheses:', [h.label for h in survivors][:8])

Synthetic checks passed. Version-space history: [{'examples_seen': 1, 'total_survivors': 12, 'pixel_survivors': 4, 'object_survivors': 8}, {'examples_seen': 2, 'total_survivors': 2, 'pixel_survivors': 2, 'object_survivors': 0}]
Surviving example hypotheses: ["pixel:geometry('anti_transpose',)", "pixel:geometry('rotate_180',)"]


# 5. Preliminary experiment on the public training tasks

Compare pixel only wiht pixel + object representations while holding the evaluation tasks fixed. The published training training task answers are not fed to solve_task and only used to score items. A task is only considered solved if every test item has the correct ouput in at least one of its two attempts. 

In [23]:
def local_evaluation(challenges, solutions, include_objects=True, max_tasks=None):
    task_ids = list(challenges)
    if max_tasks is not None:
        task_ids = task_ids[:max_tasks]
    solved_tasks = 0
    solved_items = 0
    total_items = 0
    tasks_with_consistent_rule = 0
    survivors_after_each_example = Counter()
    representation_of_first_correct = Counter()

    for task_id in tqdm(task_ids, desc='Local evaluation', leave=False):
        task = challenges[task_id]
        predictions, consistent, history = solve_task(task, include_objects)
        if consistent:
            tasks_with_consistent_rule += 1
        for stage in history:
            survivors_after_each_example[stage['examples_seen']] += stage['total_survivors']

        truths = solutions[task_id]
        assert len(predictions) == len(truths), f'Mismatched test item count for {task_id}'
        item_correct = []
        for item_index, (prediction, truth) in enumerate(zip(predictions, truths)):
            correct = any(np.array_equal(as_grid(prediction[attempt]), as_grid(truth))
                          for attempt in ('attempt_1', 'attempt_2'))
            item_correct.append(correct)
            solved_items += int(correct)
            total_items += 1
            if correct:
                # Diagnostic only: identifies one first matching rule, if present.
                for hypothesis in consistent:
                    candidate = hypothesis.apply(task['test'][item_index]['input'])
                    if candidate is not None and np.array_equal(candidate, as_grid(truth)):
                        representation_of_first_correct[hypothesis.representation] += 1
                        break

        solved_tasks += int(all(item_correct))

    total_tasks = len(task_ids)
    return {
        'tasks': total_tasks,
        'test_items': total_items,
        'tasks_solved': solved_tasks,
        'task_exact_match_rate': solved_tasks / total_tasks if total_tasks else 0.0,
        'test_items_solved': solved_items,
        'item_exact_match_rate': solved_items / total_items if total_items else 0.0,
        'tasks_with_consistent_rule': tasks_with_consistent_rule,
        'average_survivors_by_demo_index': {
            i: total / max(1, sum(len(challenges[t]['train']) >= i for t in task_ids))
            for i, total in sorted(survivors_after_each_example.items())
        },
        'first_correct_rule_representation_counts': dict(representation_of_first_correct),
    }

In [24]:
train_challenges = load_json('arc-agi_training_challenges.json')
train_solutions = load_json('arc-agi_training_solutions.json')

# Use all training tasks
MAX_PILOT_TASKS = None # or 25
pixel_only = local_evaluation(train_challenges, train_solutions, include_objects=False,
                              max_tasks=MAX_PILOT_TASKS)
pixel_and_objects = local_evaluation(train_challenges, train_solutions, include_objects=True,
                                     max_tasks=MAX_PILOT_TASKS)
print('Pixel-only baseline:')
print(json.dumps(pixel_only, indent=2))
print('\nPixel + object representation:')
print(json.dumps(pixel_and_objects, indent=2))

Pixel-only baseline:
{
  "tasks": 400,
  "test_items": 416,
  "tasks_solved": 9,
  "task_exact_match_rate": 0.0225,
  "test_items_solved": 9,
  "item_exact_match_rate": 0.021634615384615384,
  "tasks_with_consistent_rule": 9,
  "average_survivors_by_demo_index": {
    "1": 0.025,
    "2": 0.0225,
    "3": 0.023255813953488372,
    "4": 0.028037383177570093,
    "5": 0.0,
    "6": 0.0,
    "7": 0.0,
    "8": 0.0,
    "9": 0.0,
    "10": 0.0
  },
  "first_correct_rule_representation_counts": {
    "pixel": 9
  }
}

Pixel + object representation:
{
  "tasks": 400,
  "test_items": 416,
  "tasks_solved": 14,
  "task_exact_match_rate": 0.035,
  "test_items_solved": 14,
  "item_exact_match_rate": 0.03365384615384615,
  "tasks_with_consistent_rule": 16,
  "average_survivors_by_demo_index": {
    "1": 0.08,
    "2": 0.0625,
    "3": 0.05813953488372093,
    "4": 0.09345794392523364,
    "5": 0.06896551724137931,
    "6": 0.0,
    "7": 0.0,
    "8": 0.0,
    "9": 0.0,
    "10": 0.0
  },
  "first

In [25]:
solvable = []

for task_id, task in train_challenges.items():
    survivors, history = compute_version_space(
        task, include_objects=True
    )

    if survivors:
        solvable.append({
            "task_id": task_id,
            "hypotheses": len(survivors),
            "example_rule": survivors[0].label
        })

print(f"Tasks with consistent hypotheses: {len(solvable)}")
print(json.dumps(solvable[:10], indent=2))

Tasks with consistent hypotheses: 16
[
  {
    "task_id": "1cf80156",
    "hypotheses": 4,
    "example_rule": "objects_4:object('crop', 'largest', 4, None)"
  },
  {
    "task_id": "1f85a75f",
    "hypotheses": 2,
    "example_rule": "objects_4:object('crop', 'largest', 4, None)"
  },
  {
    "task_id": "23b5c85d",
    "hypotheses": 2,
    "example_rule": "objects_4:object('crop', 'smallest', 4, None)"
  },
  {
    "task_id": "39a8645d",
    "hypotheses": 1,
    "example_rule": "objects_8:object('crop', 'smallest', 8, None)"
  },
  {
    "task_id": "3c9b0459",
    "hypotheses": 1,
    "example_rule": "pixel:geometry('rotate_180',)"
  },
  {
    "task_id": "6150a2bd",
    "hypotheses": 1,
    "example_rule": "pixel:geometry('rotate_180',)"
  },
  {
    "task_id": "67a3c6ac",
    "hypotheses": 1,
    "example_rule": "pixel:geometry('flip_horizontal',)"
  },
  {
    "task_id": "68b16354",
    "hypotheses": 1,
    "example_rule": "pixel:geometry('flip_vertical',)"
  },
  {
    "task_id": 


# 6. Generate Gradescope Submission

This generates a file called `submission.json` that has the following format:

```
{"00576224": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "009d5c81": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "12997ef3": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]},
              {"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 ...
}
```


In [ ]:
test_challenges = load_json('arc-agi_test_challenges.json')
submission = {}
no_consistent_rule = 0
for task_id, task in tqdm(test_challenges.items(), desc='Generate predictions'):
    predictions, consistent, history = solve_task(task, include_objects=True)
    submission[task_id] = predictions
    no_consistent_rule += int(not consistent)

with open('submission.json', 'w') as fp:
    json.dump(submission, fp)

assert len(submission) == len(test_challenges)
assert all(len(submission[t]) == len(test_challenges[t]['test']) for t in test_challenges)
assert all(set(answer) == {'attempt_1', 'attempt_2'}
           for answers in submission.values() for answer in answers)
print(f'Wrote submission.json for {len(submission)} tasks.')
print(f'Tasks with no surviving rule (identity fallback used): {no_consistent_rule}')

Generate predictions: 100%|██████████| 400/400 [00:02<00:00, 161.35it/s]


Wrote submission.json for 400 tasks.
Tasks with no surviving rule (identity fallback used): 400


: 